# Purpose of this notebook
Step 1 of the project workflow: check data quality and split once, then lock the test set away.

**Research question:** Can physicochemical properties identify high-quality white Vinho Verde wine?

In [1]:
import numpy as np
import pandas as pd

In [2]:
wine_white = pd.read_csv("datasets/winequality-white.csv", delimiter=';')

## Why white wine only
The UCI data has separate red and white files. We model white wine only:
- **Rare-class size.** After deduplication there are 825 good white wines but only 184 good red wines. With 10-fold CV on an 80% training set, red would leave about 15 good wines per validation fold, which is too few for stable PR-AUC, sensitivity or cost estimates.
- **Different populations.** Red and white differ strongly in chemistry (for example sulphur dioxide and residual sugar). Pooling them would mix two class-conditional distributions with different covariances, which breaks LDA's equal-covariance assumption, and the share of good wines also differs (13.6% vs 21.6%).

So the research question is narrowed to: *can physicochemical properties identify high-quality white Vinho Verde wine?* The red file is read below only to compute these counts. It is not used anywhere else.

In [3]:
red = pd.read_csv("datasets/winequality-red.csv", delimiter=';')
raw_white = pd.read_csv("datasets/winequality-white.csv", delimiter=';')
counts = pd.DataFrame({
    name: {'rows': len(d), 'unique rows': len(d.drop_duplicates()),
           'share good (all rows)': round((d['quality'] >= 7).mean(), 3),
           'share good (unique)': round((d.drop_duplicates()['quality'] >= 7).mean(), 3),
           'good wines (unique)': int((d.drop_duplicates()['quality'] >= 7).sum())}
    for name, d in [('white', raw_white), ('red', red)]
})
counts

,white,red
rows,4898.000,1599.000
unique rows,3961.000,1359.000
share good (all rows),0.216,0.136
share good (unique),0.208,0.135
good wines (unique),825.000,184.000


# Data Quality Check

## Size and data types

In [4]:
wine_white.shape

(4898, 12)

In [5]:
wine_white.dtypes

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

## Null values
No missing values

In [6]:
wine_white.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

## Duplicate values

In [7]:
wine_white.duplicated().sum()

np.int64(937)

### Same wine recorded twice, or different wines?
If duplicates are repeat records of one wine, keeping them lets the same wine sit in both a training and a validation fold (or in both train and test), which inflates scores. If they were different wines that happen to measure identically, they should be kept. We check two things: whether copies ever disagree on `quality`, and how far apart copies sit in the file.

In [8]:
dup_rows = wine_white[wine_white.duplicated(keep=False)]
features = wine_white.columns.drop('quality').tolist()

# Do copies with identical measurements ever have different quality scores?
quality_per_group = dup_rows.groupby(features)['quality'].nunique()
print('Duplicate groups:', len(quality_per_group))
print('Groups whose copies disagree on quality:', (quality_per_group > 1).sum())

# How far apart (in rows) are the copies of each group?
group_id = dup_rows.groupby(features).ngroup()
gap = dup_rows.index.to_series().groupby(group_id).agg(lambda s: s.max() - s.min())
print('Median row gap between copies:', gap.median())
print('Share of groups with copies within 5 rows:', round((gap <= 5).mean(), 2))
print('Copies per group:')
print(dup_rows.groupby(group_id).size().value_counts().sort_index())

Duplicate groups: 772
Groups whose copies disagree on quality: 0
Median row gap between copies: 4.0
Share of groups with copies within 5 rows: 0.6
Copies per group:
2    673
3     61
4     24
5      6
6      4
7      2
8      2
Name: count, dtype: int64


In [9]:
# Exact duplicates on the 11 features alone = exact duplicates on all 12 columns,
# so identical measurements never carry different quality scores.
assert wine_white.duplicated(subset=features).sum() == wine_white.duplicated().sum() == 937
print('Duplicates on 11 features:', wine_white.duplicated(subset=features).sum(),
      '| on 11 features + quality:', wine_white.duplicated().sum())

Duplicates on 11 features: 937 | on 11 features + quality: 937


All 11 measurements match exactly (several to 4-5 decimal places), copies never disagree on quality, and most copies sit within a few rows of each other. We judge these to be the same wine measured more than once, not distinct wines. Keeping them as independent rows would let one wine sit in both a training and a validation fold, which inflates CV scores. Following the course feedback (*"if duplicates are the same wine measured twice, grouped CV is correct"*), we handle them in two steps:

1. **Split on unique wines (this notebook).** Duplicates are dropped *only to decide which wines go to train and which to test*, so no wine can end up on both sides.
2. **Re-attach the duplicates afterwards (`wine_utils.load_all`).** Each unique wine gets a `wine_id`, and the raw file is merged back on all 12 columns. Every repeat measurement inherits its wine's id and split (train 3,926 rows, test 972 rows, 4,898 in total). All CV uses `StratifiedGroupKFold` with `groups = wine_id`, so every copy of a wine stays in the same fold.

In [10]:
wine_white = wine_white.drop_duplicates().reset_index(drop=True)

In [11]:
wine_white.shape

(3961, 12)

## Class balance
We binarise `quality` (good = 7 or above). Only the class proportion is checked here; how predictors relate to the outcome is left to EDA on the training set.

In [12]:
print('Share of good wines (quality >= 7):', (wine_white['quality'] >= 7).mean().round(3))
wine_white['quality'].value_counts().sort_index()

Share of good wines (quality >= 7): 0.208


quality
3      20
4     153
5    1175
6    1788
7     689
8     131
9       5
Name: count, dtype: int64

About 21% of wines are good, so the classes are imbalanced. Accuracy alone would be misleading. We use stratified splitting and stratified CV, and PR curves alongside ROC curves.

We identify 937 duplicate rows (772 wines recorded 2-8 times). They are dropped here to make the split on 3,961 unique wines, then re-attached for modelling, as described above.

## Domain specific checks

### Checking number of negative values in features (expect 0)
Physicochemical features should not have negative values

In [13]:
numeric_cols = wine_white.select_dtypes(include=['number']).columns

# Identify negative values (< 0) per column
negative_counts = (wine_white[numeric_cols] < 0).sum()
negative_counts

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

### Check physiochemical features that should not be 0

In [14]:
# Flag chemically impossible zeros
impossible_zero_cols = ['density', 'pH', 'fixed acidity', 'alcohol']
impossible_zeros = (wine_white[impossible_zero_cols] == 0.0).sum()
print("\n--- Chemically Impossible Zeros ---")
print(impossible_zeros)


--- Chemically Impossible Zeros ---
density          0
pH               0
fixed acidity    0
alcohol          0
dtype: int64


# Split Into Training and Test Set
Use a 80/20 train test split, and stratification for `quality`

In [15]:
from sklearn.model_selection import train_test_split

# Split the entire DataFrame directly (keeps X and y together)
train_df, test_df = train_test_split(
    wine_white,
    test_size=0.20,
    random_state=211,
    stratify=wine_white['quality']  # Preserves target class proportions
)

# Save the train and test DataFrames to CSV files
train_df.to_csv("datasets/winequality-white-train.csv", index=False)
test_df.to_csv("datasets/winequality-white-test.csv", index=False)

## How duplicates re-enter after the split
The split above is made on the 3,961 unique wines, so it fixes which wines are in train and which are in test. Because every raw duplicate is an exact copy of one unique wine, each copy belongs to a wine that is entirely in train or entirely in test. `wine_utils.load_train()` and `load_test()` re-attach the copies (3,926 and 972 rows) and return each row's `wine_id`. Notebook 03 uses `StratifiedGroupKFold` with these ids, so all copies of a wine fall in the same fold, and it asserts that no `wine_id` appears on both sides of any fold. The CSVs written here are never regenerated or edited.